# PC-Free-T4 — Inicio rápido
Selecciona **GPU T4** y usa **Ejecutar todo** en un entorno nuevo.
Arranca solo **XFCE + aceleración T4 + RustDesk**; no espera Firefox, Battly ni Steam.
Las aplicaciones de abajo son opcionales y están desactivadas por defecto.

Colab gratuito restringe escritorio/control remoto: usa esta receta solo donde el proveedor lo permita.
No evade restricciones, no activa pagos ni monta Drive. GPU y continuidad no garantizadas.
El controlador debe ser 580.82.07; si cambia, detenerse. No reinstalar sobre una sesión activa.
Las credenciales temporales de RustDesk aparecen solo en la salida privada: no guardarlas en GitHub.


In [ ]:
SOURCES = {'00-preflight.py': "import os, subprocess, shutil, secrets, time, shlex, pwd, tarfile, urllib.request\nfrom pathlib import Path\nfrom IPython.display import display\nfrom PIL import Image as PILImage\n\nassert os.geteuid() == 0, 'Esta configuración experimental requiere el entorno Linux administrado de la prueba.'\nassert not os.path.ismount('/content/drive'), 'Desmonta Drive antes de ejecutar este instalador experimental.'\nr = subprocess.run(['nvidia-smi', '--query-gpu=name,driver_version', '--format=csv,noheader'], capture_output=True, text=True, check=True)\nname, driver_version = [x.strip() for x in r.stdout.strip().split(',')]\nprint('GPU:', name, '| Controlador:', driver_version)\nassert 'T4' in name, 'Esta receta solo fue comprobada con Tesla T4.'\nassert driver_version == '580.82.07', 'El controlador cambió. Revisar paquetes NVIDIA antes de continuar; no forzar otra versión.'\nassert shutil.disk_usage('/tmp').free > 8 * 1024**3, 'Se requieren al menos 8 GiB libres para la preparación; Minecraft puede necesitar más.'\nassert not Path('/tmp/.X11-unix/X98').exists(), 'Ya existe el escritorio :98. No reinstales encima ni reinicies la máquina; usa el entorno activo.'\nprint('Comprobaciones terminadas. Sin Drive, pagos ni medidas para evitar cortes de sesión.')\n", '01-desktop-packages.py': "base = Path('/tmp/pcfree-desktop'); base.mkdir(exist_ok=True)\npackages = ['xfce4-session', 'xfwm4', 'xfce4-panel', 'xfdesktop4', 'thunar', 'mousepad', 'xfce4-terminal', 'dbus-x11', 'mesa-utils', 'scrot', 'x11-apps', 'x11-xserver-utils', 'xauth', 'xdotool', 'wmctrl', 'curl', 'file', 'lsof', 'zenity', 'xdg-user-dirs']\nlog = base/'install.log'\nwith log.open('w') as f:\n    subprocess.run(['apt-get', 'update', '-qq'], stdout=f, stderr=subprocess.STDOUT, timeout=240, check=True)\n    subprocess.run(['apt-get', 'install', '-y', '--no-install-recommends', *packages], stdout=f, stderr=subprocess.STDOUT, timeout=480, check=True, env={**os.environ, 'DEBIAN_FRONTEND':'noninteractive'})\nprint('Escritorio instalado desde paquetes de Ubuntu.')\ndesktop_procs = []\ndef spawn_desktop(args, name):\n    log_handle = (base/(name+'.log')).open('w')\n    process = subprocess.Popen(args, env=desktop_env, stdout=log_handle, stderr=subprocess.STDOUT, start_new_session=True)\n    desktop_procs.append(process)\n    return process\n", '02-t4-libraries.py': "# Ejecutar en el cuaderno Colab: bibliotecas aisladas, sin cambiar el controlador del kernel.\nimport subprocess, os, glob, time\nfrom pathlib import Path\ngpu_root=Path('/tmp/pcfree-t4'); gpu_root.mkdir(exist_ok=True)\ngpu_log=gpu_root/'setup.log'\ngpu_downloads=['libnvidia-gl-580','libnvidia-gpucomp-580','libnvidia-cfg1-580','xserver-xorg-video-nvidia-580']\nwith gpu_log.open('w') as f:\n    r=subprocess.run(['apt-get','install','-y','--no-install-recommends','xserver-xorg-core'],stdout=f,stderr=subprocess.STDOUT,timeout=180)\n    assert r.returncode==0, gpu_log.read_text()[-2000:]\n    for pkg in gpu_downloads:\n        r=subprocess.run(['apt-get','download',pkg+'=580.82.07-0ubuntu1'],cwd=gpu_root,stdout=f,stderr=subprocess.STDOUT,timeout=180)\n        assert r.returncode==0, gpu_log.read_text()[-2000:]\n    for deb in gpu_root.glob('*.deb'):\n        subprocess.run(['dpkg-deb','-x',str(deb),str(gpu_root/'root')],check=True)\nprint('T4_LIBRARIES_EXTRACTED',gpu_log.read_text()[-2500:])\nprint('LIBS', [str(p.relative_to(gpu_root)) for p in (gpu_root/'root').rglob('*') if p.name.startswith(('libGLX_nvidia','libEGL_nvidia','nvidia_drv','libnvidia-glcore'))])\nprint('DRIVER_UNCHANGED',subprocess.getoutput('nvidia-smi --query-gpu=name,driver_version --format=csv,noheader'))\n", '03-xorg.py': '# Prueba Xorg local: sin puertos de red y sin sustituir el escritorio CPU.\nroot=gpu_root/\'root\'\nlibdirs=sorted({str(p.parent) for p in root.rglob(\'*.so*\') if p.is_file()})\ngpu_env={**os.environ,\'LD_LIBRARY_PATH\':\':\'.join(libdirs)+\':\'+os.environ.get(\'LD_LIBRARY_PATH\',\'\'),\'DISPLAY\':\':98\',\'__GLX_VENDOR_LIBRARY_NAME\':\'nvidia\'}\ndriver=next(root.rglob(\'nvidia_drv.so\'))\nmodule_path=str(driver.parent.parent)+\',/usr/lib/xorg/modules\'\nbus=subprocess.check_output([\'nvidia-smi\',\'--query-gpu=pci.bus_id\',\'--format=csv,noheader\'],text=True).strip()\nparts=bus.split(\':\'); bdf=parts[-1].split(\'.\')\nbusid=\'PCI:\'+str(int(parts[-2],16))+\':\'+str(int(bdf[0],16))+\':\'+str(int(bdf[1],16))\nconf=gpu_root/\'xorg.conf\'\nconf.write_text(\'\'\'Section "ServerFlags"\n Option "AutoAddDevices" "false"\n Option "AutoAddGPU" "false"\nEndSection\nSection "Device"\n Identifier "T4"\n Driver "nvidia"\n Option "AllowEmptyInitialConfiguration" "true"\nEndSection\nSection "Screen"\n Identifier "GPU-Screen"\n Device "T4"\n DefaultDepth 24\n SubSection "Display"\n  Depth 24\n  Virtual 1280 720\n EndSubSection\nEndSection\n\'\'\')\nconf.write_text(conf.read_text().replace(\' Driver "nvidia"\',\' Driver "nvidia"\\n BusID "\'+busid+\'"\'))\nauthority=gpu_root/\'.Xauthority\'\nauthority.touch(mode=0o600,exist_ok=True)\nauthority.chmod(0o600)\nsubprocess.run([\'xauth\',\'-f\',str(authority),\'add\',\':98\',\'MIT-MAGIC-COOKIE-1\',secrets.token_hex(16)],check=True)\ngpu_env[\'XAUTHORITY\']=str(authority)\ngpu_xorg=subprocess.Popen([\'/usr/lib/xorg/Xorg\',\':98\',\'-config\',str(conf),\'-modulepath\',module_path,\'-logfile\',str(gpu_root/\'Xorg.log\'),\'-nolisten\',\'tcp\',\'-noreset\',\'-novtswitch\',\'-sharevts\',\'-auth\',str(authority)],env=gpu_env,stdout=(gpu_root/\'xorg-stdout.log\').open(\'w\'),stderr=subprocess.STDOUT,start_new_session=True)\ntime.sleep(7)\nprint(\'XORG_STATUS\',gpu_xorg.poll())\nprint(\'XORG_LOG\',(gpu_root/\'Xorg.log\').read_text()[-11000:] if (gpu_root/\'Xorg.log\').exists() else (gpu_root/\'xorg-stdout.log\').read_text())\nr=subprocess.run([\'glxinfo\',\'-B\'],env=gpu_env,capture_output=True,text=True,timeout=30)\nprint(\'T4_OPENGL_RESULT\',r.returncode,r.stdout,r.stderr)\n\nassert gpu_xorg.poll() is None, \'Xorg no inició; revisar registros.\'\nassert r.returncode==0 and \'Tesla T4\' in r.stdout, \'OpenGL T4 no confirmado.\'\n', '04-xfce.py': "assert gpu_xorg.poll() is None, 'Xorg no sigue activo. Revisar registros; no borrar archivos.'\ngpu_home=gpu_root/'home'; gpu_home.mkdir(exist_ok=True)\ngpu_runtime=gpu_root/'runtime'; gpu_runtime.mkdir(exist_ok=True); gpu_runtime.chmod(0o700)\ndesktop_env={**gpu_env, 'HOME':str(gpu_home), 'XDG_RUNTIME_DIR':str(gpu_runtime)}\nfor key in ['SESSION_MANAGER','DBUS_SESSION_BUS_ADDRESS']:\n    desktop_env.pop(key,None)\n(gpu_home/'Desktop').mkdir(exist_ok=True)\nspawn_desktop(['dbus-run-session','--','xfce4-session'],'xfce-t4')\ntime.sleep(8)\nr=subprocess.run(['xrandr','--output','DVI-D-0','--mode','1280x720','--pos','0x0'], env=desktop_env, capture_output=True, text=True)\nif r.returncode:\n    print('No se pudo ajustar DVI-D-0 automáticamente:',r.stderr)\ngl=subprocess.run(['glxinfo','-B'],env=desktop_env,capture_output=True,text=True,check=True).stdout\nprint(gl)\nassert 'NVIDIA' in gl and 'Tesla T4' in gl, 'No se confirmó renderizado T4; detener la prueba.'\nsubprocess.run(['scrot','--overwrite',str(base/'desktop.png')],env=desktop_env,check=True)\ndisplay(PILImage.open(base/'desktop.png'))\n", '05-firefox.py': 'import os, subprocess, urllib.request, tarfile, pwd, time\nfrom pathlib import Path\nfb=Path(\'/tmp/pcfree-firefox\'); fb.mkdir(exist_ok=True)\narchive=fb/\'firefox.tar.xz\'\nurl=\'https://download.mozilla.org/?product=firefox-latest-ssl&os=linux64&lang=es-ES\'\nif not (fb/\'firefox/firefox\').exists():\n    print(\'Descargando Firefox oficial...\', flush=True)\n    urllib.request.urlretrieve(url, archive)\n    with tarfile.open(archive) as tf: tf.extractall(fb, filter=\'data\')\n    archive.unlink()\nuser=\'pcfree-browser\'\ntry: account=pwd.getpwnam(user)\nexcept KeyError:\n    subprocess.run([\'useradd\',\'--create-home\',\'--home-dir\',str(fb/\'home\'),\'--shell\',\'/bin/bash\',user],check=True)\n    account=pwd.getpwnam(user)\nhome=Path(account.pw_dir)\nhome.mkdir(exist_ok=True)\nos.chown(home,account.pw_uid,account.pw_gid)\nruntime=fb/\'runtime\'; runtime.mkdir(exist_ok=True); runtime.chmod(0o700)\nos.chown(runtime,account.pw_uid,account.pw_gid)\ngrant=subprocess.run([\'xhost\',\'+SI:localuser:\'+user],env=desktop_env,capture_output=True,text=True)\nprint(\'XAUTH\',grant.returncode,grant.stdout,grant.stderr,flush=True)\nif grant.returncode: raise RuntimeError(\'No se pudo autorizar el navegador al escritorio\')\nbrowser_env={**desktop_env,\'HOME\':str(home),\'USER\':user,\'LOGNAME\':user,\'XDG_RUNTIME_DIR\':str(runtime)}\nfor key in [\'DBUS_SESSION_BUS_ADDRESS\',\'SESSION_MANAGER\',\'XAUTHORITY\']:\n    browser_env.pop(key,None)\nlog=open(fb/\'firefox.log\',\'w\')\nfirefox_process=subprocess.Popen([\'runuser\',\'-u\',user,\'--\',\'dbus-run-session\',\'--\',str(fb/\'firefox/firefox\'),\'--no-remote\',\'https://www.minecraft.net/download\'],env=browser_env,stdout=log,stderr=subprocess.STDOUT)\ntime.sleep(12)\nprint(\'FIREFOX_PROCESS\',firefox_process.pid,firefox_process.poll())\nprint(\'FIREFOX_LOG\',(fb/\'firefox.log\').read_text()[-2500:])\nif shutil.which(\'wmctrl\'): print(subprocess.run([\'wmctrl\',\'-l\'],env=desktop_env,capture_output=True,text=True).stdout)\nprint(\'FIREFOX_SETUP_DONE; DRIVE_MOUNTED\',os.path.ismount(\'/content/drive\'))\n\nimport shlex\nlauncher=Path(\'/usr/local/bin/pcfree-firefox\')\nkept={k:browser_env[k] for k in [\'DISPLAY\',\'HOME\',\'USER\',\'LOGNAME\',\'XDG_RUNTIME_DIR\',\'LD_LIBRARY_PATH\',\'__GLX_VENDOR_LIBRARY_NAME\'] if k in browser_env}\nlauncher.write_text(\'#!/bin/sh\\nexec runuser -u pcfree-browser -- env \'+ \' \'.join(shlex.quote(k+\'=\'+v) for k,v in kept.items())+\' dbus-run-session -- /tmp/pcfree-firefox/firefox/firefox "$@"\\n\')\nlauncher.chmod(0o755)\ndesk=Path(desktop_env[\'HOME\'])/\'Desktop\';desk.mkdir(exist_ok=True)\nshortcut=desk/\'Firefox.desktop\'\nshortcut.write_text(\'[Desktop Entry]\\nVersion=1.0\\nType=Application\\nName=Firefox\\nExec=/usr/local/bin/pcfree-firefox %u\\nIcon=/tmp/pcfree-firefox/firefox/browser/chrome/icons/default/default128.png\\nTerminal=false\\n\')\nshortcut.chmod(0o755)\nif shutil.which(\'gio\'):subprocess.run([\'gio\',\'set\',str(shortcut),\'metadata::trusted\',\'true\'],env=desktop_env,capture_output=True)\nsubprocess.run([\'scrot\',\'--overwrite\',\'/tmp/firefox-check.png\'],env=desktop_env,check=True)\ndisplay(PILImage.open(\'/tmp/firefox-check.png\'))\nprint(\'FIREFOX_SHORTCUT_READY\')\n', '06-battly.py': '# Run inside the existing Colab runtime; not in the agent sandbox.\nimport urllib.request, hashlib, tarfile, os\nfrom pathlib import Path\nbattly_base=Path(\'/tmp/pcfree-battly\'); battly_base.mkdir(exist_ok=True)\nbattly_archive=battly_base/\'battly.tar.gz\'\nbattly_url=\'https://github.com/BattlyLauncher/battlylauncher/releases/download/3.0.1/Battly-Launcher-linux-x64.tar.gz\'\nbattly_sha=\'547a282d28020b71b14a486f25e7190a3e1ff38fb89963e6c8af08a905b51a01\'\nprint(\'Descargando Battly 3.0.1 oficial...\',flush=True)\nurllib.request.urlretrieve(battly_url,battly_archive)\nactual=hashlib.file_digest(battly_archive.open(\'rb\'),\'sha256\').hexdigest()\nassert actual==battly_sha,\'SHA256 no coincide; no ejecutar\'\nprint(\'SHA256 correcto\',flush=True)\nbattly_root=battly_base/\'app\';battly_root.mkdir(exist_ok=True)\nwith tarfile.open(battly_archive) as tf:tf.extractall(battly_root,filter=\'data\')\nbattly_archive.unlink()\nprint(\'Archivos principales:\',[(str(p.relative_to(battly_root)),oct(p.stat().st_mode&0o777)) for p in battly_root.iterdir()])\nprint(\'Ejecutables:\',[str(p) for p in battly_root.rglob(\'*\') if p.is_file() and os.access(p,os.X_OK)][:35])\nprint(\'BATTLY_DOWNLOAD_COMPLETE\')\nimport shlex, subprocess, time\nbattly_bin=next(battly_root.rglob(\'battlylauncher\'))\nprint(\'DEPENDENCIAS\',subprocess.getoutput(\'ldd \'+shlex.quote(str(battly_bin))))\nsandbox=battly_bin.parent/\'chrome-sandbox\'\nif sandbox.exists():\n    os.chown(sandbox,0,0)\n    sandbox.chmod(0o4755)\nbl=Path(\'/usr/local/bin/pcfree-battly\')\nkept={k:browser_env[k] for k in [\'DISPLAY\',\'HOME\',\'USER\',\'LOGNAME\',\'XDG_RUNTIME_DIR\',\'LD_LIBRARY_PATH\',\'__GLX_VENDOR_LIBRARY_NAME\'] if k in browser_env}\nbl.write_text(\'#!/bin/sh\\nexec runuser -u pcfree-browser -- env \'+ \' \'.join(shlex.quote(k+\'=\'+v) for k,v in kept.items())+\' dbus-run-session -- \'+shlex.quote(str(battly_bin))+\' "$@"\\n\')\nbl.chmod(0o755)\nicons=list((battly_bin.parent/\'resources/app\').rglob(\'icon.png\'))\nicon=str(icons[0]) if icons else \'applications-games\'\nshortcut=Path(desktop_env[\'HOME\'])/\'Desktop/Battly.desktop\'\nshortcut.write_text(\'[Desktop Entry]\\nVersion=1.0\\nType=Application\\nName=Battly Launcher\\nExec=/usr/local/bin/pcfree-battly\\nIcon=\'+icon+\'\\nTerminal=false\\nCategories=Game;\\n\')\nshortcut.chmod(0o755)\nsubprocess.run([\'gio\',\'set\',str(shortcut),\'metadata::trusted\',\'true\'],env=desktop_env,capture_output=True)\nbattly_log=open(battly_base/\'battly.log\',\'w\')\nbattly_process=subprocess.Popen([str(bl)],env=desktop_env,stdout=battly_log,stderr=subprocess.STDOUT)\ntime.sleep(10)\nprint(\'BATTLY_PROCESS\',battly_process.pid,battly_process.poll())\nprint(\'BATTLY_LOG\',(battly_base/\'battly.log\').read_text()[-2500:])\nsubprocess.run([\'scrot\',\'--overwrite\',\'/tmp/battly-check.png\'],env=desktop_env,check=True)\ndisplay(PILImage.open(\'/tmp/battly-check.png\'))\nprint(\'BATTLY_SHORTCUT_DONE\')\n\nappdir=battly_bin.parent/\'resources/app\'\n# Run in the existing Colab runtime after battly-setup.py.\n# TAR Linux build: AppImage updater returns without event, leaving splash stuck.\nimport signal, time\nsource=appdir/\'src/app.js\'\nbackup=source.with_suffix(\'.js.original\')\nif not backup.exists():backup.write_bytes(source.read_bytes())\ntext=source.read_text()\nold=\'if (dev || !app.isPackaged) {\'\nnew=\'if (dev || !app.isPackaged || (process.platform === "linux" && !process.env.APPIMAGE)) {\'\nif old in text:\n    assert text.count(old)==1\n    source.write_text(text.replace(old,new,1))\nelse:\n    assert new in text,\'Unexpected updater code; do not patch blindly\'\nfor d in Path(\'/proc\').iterdir():\n    if not d.name.isdigit():continue\n    try:\n        if os.path.realpath(str(d/\'exe\'))==str(battly_bin):os.kill(int(d.name),signal.SIGTERM)\n    except (OSError,ProcessLookupError):pass\ntime.sleep(3)\nbattly_log=open(battly_base/\'battly-fixed.log\',\'w\')\nbattly_process=subprocess.Popen([str(bl)],env=desktop_env,stdout=battly_log,stderr=subprocess.STDOUT)\ntime.sleep(15)\nprint(\'BATTLY_PATCH_PROCESS\',battly_process.pid,battly_process.poll())\nprint((battly_base/\'battly-fixed.log\').read_text()[-3500:])\nsubprocess.run([\'scrot\',\'--overwrite\',\'/tmp/battly-fixed.png\'],env=desktop_env,check=True)\ndisplay(PILImage.open(\'/tmp/battly-fixed.png\'))\nprint(\'BATTLY_PATCH_CHECK_DONE\')\n', '07-rustdesk.py': "import os, subprocess, time, urllib.request\nfrom pathlib import Path\nprint('RUSTDESK_PREFLIGHT')\nassert not os.path.ismount('/content/drive'), 'Drive debe permanecer desconectado'\nassert 'desktop_env' in globals(), 'El entorno grafico ya no esta disponible'\nr = subprocess.run(['xrandr'], env=desktop_env, capture_output=True, text=True)\nassert r.returncode == 0, 'La pantalla grafica no responde'\nprint('DISPLAY_ALIVE', desktop_env.get('DISPLAY'))\nrd_base = Path('/tmp/pcfree-rustdesk'); rd_base.mkdir(exist_ok=True)\nrd_deb = rd_base/'rustdesk.deb'\nurl = 'https://github.com/rustdesk/rustdesk/releases/download/1.4.4/rustdesk-1.4.4-x86_64.deb'\nurllib.request.urlretrieve(url, rd_deb)\nprint('OFFICIAL_PACKAGE_BYTES', rd_deb.stat().st_size)\nprint('PACKAGE', subprocess.check_output(['dpkg-deb','-f',str(rd_deb),'Package','Version','Depends'],text=True))\nrd_root = rd_base/'root'\nsubprocess.run(['dpkg-deb','-x',str(rd_deb),str(rd_root)],check=True)\nrd_bin = rd_root/'usr/share/rustdesk/rustdesk'\nrd_env = {**desktop_env, 'LD_LIBRARY_PATH':str(rd_bin.parent/'lib')+':'+desktop_env.get('LD_LIBRARY_PATH','')}\ndeps = ['libgstreamer-plugins-base1.0-0','libxdo3','libva2','libva-drm2','libva-x11-2','libayatana-appindicator3-1']\nwith (rd_base/'deps.log').open('w') as log:\n    subprocess.run(['apt-get','install','-y','--no-install-recommends',*deps],stdout=log,stderr=subprocess.STDOUT,env={**os.environ,'DEBIAN_FRONTEND':'noninteractive'},check=True)\nprint('VERSION', subprocess.run([str(rd_bin),'--version'],env=rd_env,capture_output=True,text=True,timeout=20))\n# Execute only once; do not repeat while these processes remain active.\nrd_server = subprocess.Popen([str(rd_bin),'--server'],env=rd_env,cwd=str(rd_bin.parent),stdout=(rd_base/'server.log').open('w'),stderr=subprocess.STDOUT,start_new_session=True)\nrd_gui = subprocess.Popen(['dbus-run-session','--',str(rd_bin)],env=rd_env,cwd=str(rd_bin.parent),stdout=(rd_base/'gui.log').open('w'),stderr=subprocess.STDOUT,start_new_session=True)\ntime.sleep(8)\nprint('RUSTDESK_STATUS',rd_server.poll(),rd_gui.poll())\nprint('ID',subprocess.run([str(rd_bin),'--get-id'],env=rd_env,capture_output=True,text=True,timeout=15).stdout.strip())\nsubprocess.run(['scrot','--overwrite',str(rd_base/'screen.png')],env=desktop_env,check=True)\nfrom PIL import Image as PILImage\ndisplay(PILImage.open(rd_base/'screen.png'))\n", '08-steam.py': 'assert \'browser_env\' in globals(), \'Preparar primero el usuario de aplicaciones con Firefox.\'\nsteam_base=Path(\'/tmp/pcfree-steam\'); steam_base.mkdir(exist_ok=True)\nsteam_deb=steam_base/\'steam.deb\'\nurllib.request.urlretrieve(\'https://repo.steampowered.com/steam/archive/stable/steam_latest.deb\',steam_deb)\nprint(subprocess.run([\'dpkg-deb\',\'-f\',str(steam_deb),\'Version\'],capture_output=True,text=True,check=True).stdout)\nsteam_root=steam_base/\'root\';steam_root.mkdir(exist_ok=True)\nsubprocess.run([\'dpkg-deb\',\'-x\',str(steam_deb),str(steam_root)],check=True)\nsubprocess.run([\'dpkg\',\'--add-architecture\',\'i386\'],check=True)\nlog=steam_base/\'libraries.log\'\nwith log.open(\'w\') as f:\n    subprocess.run([\'apt-get\',\'update\',\'-qq\'],stdout=f,stderr=subprocess.STDOUT,check=True,timeout=240)\n    subprocess.run([\'apt-get\',\'install\',\'-y\',\'--no-install-recommends\',\'lib32gcc-s1\',\'lib32stdc++6\',\'libgl1:i386\',\'libdrm2:i386\'],stdout=f,stderr=subprocess.STDOUT,check=True,timeout=480,env={**os.environ,\'DEBIAN_FRONTEND\':\'noninteractive\'})\n# Official launcher supports running directly beside its bootstrap tarball.\n# Dependencies are handled above; no interactive package helper is necessary.\nsteam_script=steam_root/\'usr/lib/steam/bin_steam.sh\'\nassert steam_script.exists()\nkept={k:browser_env[k] for k in [\'DISPLAY\',\'HOME\',\'USER\',\'LOGNAME\',\'XDG_RUNTIME_DIR\',\'LD_LIBRARY_PATH\',\'__GLX_VENDOR_LIBRARY_NAME\'] if k in browser_env}\nsteam_wrapper=Path(\'/usr/local/bin/pcfree-steam\')\n# Steam bootstrap may exit after handing off to a detached client. A bus tied\n# to that bootstrap via dbus-run-session can then die under steamwebhelper.\n# Keep an authenticated private session bus for the application user instead.\nsteam_session=steam_base/\'start-session.sh\'\nsteam_session.write_text(\'#!/bin/sh\\nset -eu\\nbus_address="unix:path=$XDG_RUNTIME_DIR/steam-session-bus"\\n\'\n    \'if ! dbus-send --bus="$bus_address" --type=method_call --print-reply --dest=org.freedesktop.DBus /org/freedesktop/DBus org.freedesktop.DBus.ListNames >/dev/null 2>&1; then\\n\'\n    \'  dbus-daemon --session --fork --address="$bus_address"\\nfi\\n\'\n    \'export DBUS_SESSION_BUS_ADDRESS="$bus_address"\\nexec \'+shlex.quote(str(steam_script))+\' -cef-disable-gpu "$@"\\n\')\nsteam_session.chmod(0o755)\nsteam_wrapper.write_text(\'#!/bin/sh\\nexec runuser -u pcfree-browser -- env \'+\' \'.join(shlex.quote(k+\'=\'+v) for k,v in kept.items())+\' \'+shlex.quote(str(steam_session))+\' "$@"\\n\')\nsteam_wrapper.chmod(0o755)\nshortcut=Path(desktop_env[\'HOME\'])/\'Desktop/Steam.desktop\'\nshortcut.write_text(\'[Desktop Entry]\\nVersion=1.0\\nType=Application\\nName=Steam\\nExec=/usr/local/bin/pcfree-steam\\nIcon=\'+str(steam_root/\'usr/share/icons/hicolor/256x256/apps/steam.png\')+\'\\nTerminal=false\\nCategories=Game;\\n\')\nshortcut.chmod(0o755)\nsubprocess.run([\'gio\',\'set\',str(shortcut),\'metadata::trusted\',\'true\'],env=desktop_env,capture_output=True)\nsteam_process=subprocess.Popen([str(steam_wrapper)],env=desktop_env,stdout=(steam_base/\'steam.log\').open(\'w\'),stderr=subprocess.STDOUT)\nprint(\'Steam solicitado. La primera actualización puede tardar varios minutos.\')\nprint(\'Iniciar sesión manualmente desde RustDesk, sin enviar contraseñas al chat. Steam Link aún no validado.\')\n', '09-status.py': "print('Drive montado:',os.path.ismount('/content/drive'))\nprint(subprocess.run(['nvidia-smi','--query-gpu=name,driver_version','--format=csv,noheader'],capture_output=True,text=True).stdout)\nprint(subprocess.run(['glxinfo','-B'],env=desktop_env,capture_output=True,text=True).stdout)\nprint('Espacio temporal libre (GiB):',round(shutil.disk_usage('/tmp').free/1024**3,1))\nsubprocess.run(['scrot','--overwrite',str(base/'current.png')],env=desktop_env,check=True)\ndisplay(PILImage.open(base/'current.png'))\nprint('La captura puede contener contraseña temporal de RustDesk. No guardar salidas del cuaderno en GitHub.')\n"}
ORDER = ['00-preflight.py', '01-desktop-packages.py', '02-t4-libraries.py', '03-xorg.py', '04-xfce.py', '05-firefox.py', '06-battly.py', '07-rustdesk.py', '08-steam.py']
COMPLETED = set()
def run_stage(name):
    if name in COMPLETED:
        print('Etapa ya ejecutada en esta sesión:',name)
        return
    index=ORDER.index(name)
    # RustDesk needs the desktop, not downloads of Firefox, Battly or Steam.
    prerequisites = {
        '05-firefox.py': ['04-xfce.py'],
        '06-battly.py': ['05-firefox.py'],
        '07-rustdesk.py': ['04-xfce.py'],
        '08-steam.py': ['05-firefox.py'],
    }.get(name, ORDER[:index])
    missing=[previous for previous in prerequisites if previous not in COMPLETED]
    assert not missing, 'Completar primero: '+', '.join(missing)
    exec(compile(SOURCES[name],name,'exec'),globals())
    COMPLETED.add(name)
    print('Etapa terminada:',name)


In [ ]:
# Minimal startup; preserve all hardware/security checks.
import time as _startup_time
_startup_started = _startup_time.monotonic()
for _stage in [*ORDER[:5], '07-rustdesk.py']:
    run_stage(_stage)
print('Escritorio y RustDesk preparados en', round(_startup_time.monotonic()-_startup_started, 1), 'segundos.')
print('Conecta con los datos de RustDesk mostrados arriba. Apps opcionales abajo.')


## Aplicaciones opcionales — después de conectar
Cambia a `True` solo lo que necesites y ejecuta esta celda.
Battly y Steam requieren el usuario de aplicaciones preparado por Firefox.
La instalación opcional no reinicia ni reconstruye el escritorio.
Para respaldos manuales usa las celdas de exportación/restauración del cuaderno completo.


In [ ]:
INSTALL_FIREFOX = False
INSTALL_BATTLY = False
INSTALL_STEAM = False
if INSTALL_FIREFOX or INSTALL_BATTLY or INSTALL_STEAM:
    run_stage('05-firefox.py')
if INSTALL_BATTLY:
    run_stage('06-battly.py')
if INSTALL_STEAM:
    run_stage('08-steam.py')
